# Bronze Layer - Ingestion and Validation
This notebook handles the bronze layer of the healthcare lakehouse:
1. Ingest raw Synthea CSVs from Unity Catalog volumes (5 states) into Delta tables
2. Validate the loaded tables before anything downstream consumes them
- The bronze rule is simple: keep the raw data as-is. No cleaning, no deletions, no transformations. That work happens in the silver layer.
- The validationsection just records what we found so we know what to expect downstream.
- Source data: Synthea-generated synthetic patient records for California, Florida, New York, Pennsylvania, and Texas.
- Target schema: `healthcare_demo.healthcare_bronze`

## Part 1 - Ingestion
Load each table from all 5 state volumes, tag rows with `source_state`, and union them into a single bronze table per entity.

In [0]:
from pyspark.sql.functions import lit

catalog = "healthcare_demo"
schema = "healthcare_bronze"

# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# Volume name -> human-readable state label (used in source_state column)
states = {
    "california": "California",
    "florida": "Florida",
    "new_york": "New York",
    "pennsylvania": "Pennsylvania",
    "texas": "Texas"
}

# Tables to ingest - core entities first, then clinical / financial
tables = [
    "claims",
    "devices",
    "organizations",
    "payer_transitions",
    "payers",
    "providers",
    "supplies",
    "patients",
    "encounters",
    "conditions",
    "observations",
    "medications",
    "procedures",
    "immunizations",
    "allergies",
    "careplans"
]

for table in tables:
    dfs = []

    # Read each state's copy of this table, tag it with the source state
    for volume_name, state_name in states.items():
        path = f"/Volumes/{catalog}/{schema}/{volume_name}/{state_name}/{table}.csv"

        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(path)
            .withColumn("source_state", lit(state_name))
        )

        dfs.append(df)

    # Union all 5 state DataFrames - allowMissingColumns handles schema drift
    combined_df = dfs[0]
    for df in dfs[1:]:
        combined_df = combined_df.unionByName(df, allowMissingColumns=True)

    # Overwrite the bronze table - bronze is fully reproducible from source
    combined_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(
        f"{catalog}.{schema}.{table}"
    )

## Part 2 - Validation
Now that the bronze tables are loaded, run a series of checks to confirm the data landed correctly.
- All check results are persisted to`_metadata.validation_runs` and `_metadata.validation_results`. So we have a historical record of every run.

- Checks performed:
1. Row counts per table
2. Null checks on key columns
3. Observations with null ENCOUNTER (investigation)
4. Payer_transitions with null MEMBERID (investigation)
5. Duplicate primary key checks
6. Patient foreign key integrity
7. Duplicate payer investigation
8. Duplicate organization investigation
- Bronze rule reminder: This notebook does not delete or modify any records. We only inspect. Anything that needs fixing gets handled in silver.

### Validation run setup
Initialize a Validation Run object. At the end of the notebook we commit theresults and assert that no error-level checks failed. That assertion is whatblocks downstream notebooks from running on bad data.

In [0]:
import sys
sys.path.append('/Workspace/Shared/healthcare_demo')
from validation_helper import ValidationRun

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

catalog_name = "healthcare_demo"
bronze_schema = "healthcare_bronze"
bronze_prefix = f"{catalog_name}.{bronze_schema}"

run = ValidationRun(
    spark=spark,
    catalog=catalog_name,
    notebook_name="02_bronze_validation",
    layer="bronze"
)

print(f"Validation run started: {run.run_id}")

Validation run started: df777604-3d36-4aa5-b7cf-34e71e37c292


In [0]:
# Pull the list of bronze tables straight from the catalog so the checks
# stay in sync with whatever was just ingested
tables = [
    table.name
    for table in spark.catalog.listTables(bronze_prefix)
]

print(f"Number of bronze tables found: {len(tables)}")
print(tables)

Number of bronze tables found: 16
['allergies', 'careplans', 'claims', 'conditions', 'devices', 'encounters', 'immunizations', 'medications', 'observations', 'organizations', 'patients', 'payer_transitions', 'payers', 'procedures', 'providers', 'supplies']


### 1. Row count validation
Confirm every bronze table exists and has rows. 

A zero-row bronze table almostalways means the source CSV path was wrong or the volume is empty.

In [0]:
profile_rows = []

for table in tables:
    df = spark.table(f"{bronze_prefix}.{table}")
    profile_rows.append((table, df.count(), len(df.columns)))

profile_df = spark.createDataFrame(
    profile_rows,
    ["table_name", "row_count", "column_count"]
)

display(profile_df.orderBy("table_name"))

# Record row count results
for row in profile_df.collect():
    table_name = row["table_name"]
    row_count = row["row_count"]
    is_passing = (row_count > 0)

    run.add_result(
        check_id=f"bronze.{table_name}.row_count",
        check_name=f"Row count for {table_name}",
        check_category="row_count",
        target_table=table_name,
        severity="info",
        expected=">0",
        actual=str(row_count),
        status="passed" if is_passing else "failed",
        message="" if is_passing else "Table has zero rows"
    )

print(f"Recorded {profile_df.count()} row count checks.")

table_name,row_count,column_count
allergies,21684,16
careplans,55029,10
claims,1539911,32
conditions,528897,8
devices,79953,8
encounters,827172,16
immunizations,170384,7
medications,712739,14
observations,9460263,10
organizations,15449,12


Recorded 16 row count checks.


### 2. Key column null checks
Walk through each table and count nulls in the columns we care about including primary keys, patient FKs, encounter FKs. Some nulls are expected (and investigated below), but nulls in true PKs would be a real problem.

In [0]:
key_checks = {
    "patients": ["Id"],
    "careplans": ["Id", "PATIENT", "ENCOUNTER"],
    "organizations": ["Id"],
    "allergies": ["PATIENT", "ENCOUNTER"],
    "devices": ["PATIENT", "ENCOUNTER"],
    "immunizations": ["PATIENT", "ENCOUNTER"],
    "payer_transitions": ["PATIENT", "MEMBERID", "PAYER"],
    "payers": ["Id"],
    "providers": ["Id"],
    "supplies": ["PATIENT", "ENCOUNTER"],
    "encounters": ["Id", "PATIENT"],
    "conditions": ["PATIENT", "ENCOUNTER"],
    "observations": ["PATIENT", "ENCOUNTER"],
    "medications": ["PATIENT", "ENCOUNTER"],
    "procedures": ["PATIENT", "ENCOUNTER"],
    "claims": ["Id", "PATIENTID"],
}

null_check_rows = []

for table, cols in key_checks.items():
    df = spark.table(f"{bronze_prefix}.{table}")
    
    for col_name in cols:
        null_count = df.filter(col(col_name).isNull()).count()
        null_check_rows.append((table, col_name, null_count))

null_check_df = spark.createDataFrame(
    null_check_rows,
    ["table_name", "column_name", "null_count"]
)

display(null_check_df.orderBy("table_name", "column_name"))

# Columns where nulls are critical — FKs, PKs, or other required keys
critical_null_columns = {
    ("patients", "Id"),
    ("encounters", "Id"),
    ("encounters", "PATIENT"),
    ("organizations", "Id"),
    ("providers", "Id"),
    ("payers", "Id"),
}

# Record null count results
for row in null_check_df.collect():
    table_name = row["table_name"]
    column_name = row["column_name"]
    null_count = row["null_count"]
    is_critical = (table_name, column_name) in critical_null_columns

    if is_critical:
        severity = "error"
        status = "passed" if null_count == 0 else "failed"
        msg = "" if null_count == 0 else f"{null_count} nulls in critical key column"
    else:
        severity = "warning"
        status = "passed" if null_count == 0 else "warned"
        msg = "" if null_count == 0 else f"{null_count} nulls (documented as acceptable)"

    run.add_result(
        check_id=f"bronze.{table_name}.null.{column_name}",
        check_name=f"Null count for {table_name}.{column_name}",
        check_category="null_check",
        target_table=table_name,
        target_column=column_name,
        severity=severity,
        expected="0",
        actual=str(null_count),
        status=status,
        message=msg
    )

print(f"Recorded {null_check_df.count()} null count checks.")

table_name,column_name,null_count
allergies,ENCOUNTER,0
allergies,PATIENT,0
careplans,ENCOUNTER,0
careplans,Id,0
careplans,PATIENT,0
claims,Id,0
claims,PATIENTID,0
conditions,ENCOUNTER,0
conditions,PATIENT,0
devices,ENCOUNTER,0


Recorded 30 null count checks.


Result:
- `observations.ENCOUNTER`: 316,980 nulls
- `payer_transitions.MEMBERID`: 228,151 nulls

Both warrant a closer look before deciding what to do in silver. See the next two sections.

### 3. Investigate observations with null ENCOUNTER
Some observations in Synthea are recorded at the patient level rather than inside a specific encounter. Group the null-encounter rows by category and description to see what's going on.

In [0]:
observations_df = spark.table(f"{bronze_prefix}.observations")

null_encounter_observations_df = (
    observations_df
    .filter(col("ENCOUNTER").isNull())
    .groupBy("CATEGORY", "DESCRIPTION")
    .count()
    .orderBy(col("count").desc())
)

display(null_encounter_observations_df)

null_encounter_count = (
    spark.table(f"{bronze_prefix}.observations")
    .filter(col("ENCOUNTER").isNull())
    .count()
)

run.add_result(
    check_id="bronze.observations.null_encounter",
    check_name="Observations with null ENCOUNTER",
    check_category="null_check",
    target_table="observations",
    target_column="ENCOUNTER",
    severity="warning",
    expected="0 (but acceptable >0 due to QALY/DALY/QOLS records)",
    actual=str(null_encounter_count),
    status="warned" if null_encounter_count > 0 else "passed",
    message=f"{null_encounter_count} observations without an encounter — documented as QALY/DALY/QOLS lifetime metrics"
)

print(f"Recorded observations null encounter check: {null_encounter_count} nulls.")

CATEGORY,DESCRIPTION,count
null,QALY,105660
null,DALY,105660
null,QOLS,105660


Recorded observations null encounter check: 316980 nulls.


All the null-encounter observations fall into three categories:
- QALY: Quality-Adjusted Life Years
- DALY: Disability-Adjusted Life Years
- and QOLS: Quality of Life Score

These are lifetime patient health metrics, not encounter events

Decision: Leave them in bronze. They're meaningful records, just not encounter-tied.

### 4. Investigate payer_transitions with null MEMBERID
MEMBERID is the insurance member number. Some payer transition records won't have one typically self-pay enrollments where there's no insurance plan to generate a member ID. Confirm that PATIENT and PAYER are still populated.

In [0]:
from pyspark.sql.functions import col

df = spark.table("healthcare_demo.healthcare_bronze.payer_transitions")

print(
    "PATIENT nulls:",
    df.filter(col("PATIENT").isNull()).count()
)

print(
    "PAYER nulls:",
    df.filter(col("PAYER").isNull()).count()
)

null_memberid_count = (
    spark.table(f"{bronze_prefix}.payer_transitions")
    .filter(col("MEMBERID").isNull())
    .count()
)

run.add_result(
    check_id="bronze.payer_transitions.null_memberid",
    check_name="payer_transitions with null MEMBERID",
    check_category="null_check",
    target_table="payer_transitions",
    target_column="MEMBERID",
    severity="warning",
    expected="0 (but acceptable >0 due to self-pay)",
    actual=str(null_memberid_count),
    status="warned" if null_memberid_count > 0 else "passed",
    message=f"{null_memberid_count} payer transition records without a MEMBERID — documented as self-pay enrollments"
)

print(f"Recorded payer_transitions null memberid check: {null_memberid_count} nulls.")

PATIENT nulls: 0
PAYER nulls: 0
Recorded payer_transitions null memberid check: 228151 nulls.


payer_transitions.MEMBERID has 228,151 nulls.

PATIENT and PAYER are fully populated, so the records are still usable. MEMBERID just isn't always present (self-pay enrollments). Bronze keeps the records unchanged. Silver can decide whether MEMBERID is needed downstream.

### 5. Duplicate primary key checks
For tables with an `Id` column, check uniqueness within each `source_state`. A duplicate `(source_state, Id)` pair would mean the same source file was loaded twice, or the CSV itself was corrupted. Cross-state duplicates of `Id` are a separate story and get investigated in sections 7 and 8.

In [0]:
primary_keys = {
    "patients": "Id",
    "encounters": "Id",
    "claims": "Id",
    "organizations": "Id",
    "providers": "Id",
    "payers": "Id",
    "careplans": "Id",
}

duplicate_check_rows = []

for table, pk in primary_keys.items():
    df = spark.table(f"{bronze_prefix}.{table}")

    duplicate_key_count = (
        df.groupBy("source_state", pk)
          .count()
          .filter(col("count") > 1)
          .count()
    )

    duplicate_check_rows.append((table, pk, duplicate_key_count))

duplicate_check_df = spark.createDataFrame(
    duplicate_check_rows,
    ["table_name", "key_column", "duplicate_key_count"]
)

display(duplicate_check_df.orderBy("table_name"))

# Record duplicate key results — duplicates within (source_state, Id) are errors
for row in duplicate_check_df.collect():
    table_name = row["table_name"]
    duplicate_count = row["duplicate_key_count"]
    is_passing = (duplicate_count == 0)

    run.add_result(
        check_id=f"bronze.{table_name}.pk_uniqueness",
        check_name=f"PK uniqueness within (source_state, Id) for {table_name}",
        check_category="pk_uniqueness",
        target_table=table_name,
        target_column="(source_state, Id)",
        severity="error",
        expected="0",
        actual=str(duplicate_count),
        status="passed" if is_passing else "failed",
        message="" if is_passing else f"{duplicate_count} duplicate composite keys — indicates load corruption"
    )

print(f"Recorded {duplicate_check_df.count()} PK uniqueness checks.")

table_name,key_column,duplicate_key_count
careplans,Id,0
claims,Id,0
encounters,Id,0
organizations,Id,0
patients,Id,0
payers,Id,0
providers,Id,0


Recorded 7 PK uniqueness checks.


All PKs are unique within their source state. The organizations and payerstables show cross-state duplicate IDs (13 and 10 respectively) - these are thesame entity appearing in multiple state files. See sections 7 and 8.

### 6. Patient foreign key integrity
Make sure every PATIENT reference in the clinical tables actually exists in the patients table. Orphan FKs would mean we lost patient records during ingestion or the source data itself is broken.

In [0]:
patients_df = (
    spark.table(f"{bronze_prefix}.patients")
    .select("Id")
    .distinct()
)

patient_fk_tables = {
    "encounters": "PATIENT",
    "conditions": "PATIENT",
    "observations": "PATIENT",
    "medications": "PATIENT",
    "procedures": "PATIENT",
    "immunizations": "PATIENT",
    "allergies": "PATIENT",
    "careplans": "PATIENT",
    "devices": "PATIENT",
    "supplies": "PATIENT"
}

orphan_check_rows = []

for table, fk in patient_fk_tables.items():
    df = spark.table(f"{bronze_prefix}.{table}")

    orphan_count = (
        df.join(
            patients_df,
            df[fk] == patients_df["Id"],
            "left_anti"
        )
        .count()
    )

    orphan_check_rows.append((table, fk, orphan_count))

orphan_check_df = spark.createDataFrame(
    orphan_check_rows,
    ["table_name", "patient_fk_column", "orphan_patient_record_count"]
)

display(orphan_check_df.orderBy("table_name"))

# Record each FK check result for persistence and assertion
for row in orphan_check_df.collect():
    table_name = row["table_name"]
    fk_column = row["patient_fk_column"]
    orphan_count = row["orphan_patient_record_count"]
    is_passing = (orphan_count == 0)

    run.add_result(
        check_id=f"bronze.{table_name}.fk_patient",
        check_name=f"Patient FK integrity for {table_name}",
        check_category="fk_integrity",
        target_table=table_name,
        target_column=fk_column,
        severity="error",
        expected="0",
        actual=str(orphan_count),
        status="passed" if is_passing else "failed",
        message="" if is_passing else f"{orphan_count} orphan patient FK references"
    )

print(f"Recorded {len(orphan_check_rows)} patient FK checks.")

table_name,patient_fk_column,orphan_patient_record_count
allergies,PATIENT,0
careplans,PATIENT,0
conditions,PATIENT,0
devices,PATIENT,0
encounters,PATIENT,0
immunizations,PATIENT,0
medications,PATIENT,0
observations,PATIENT,0
procedures,PATIENT,0
supplies,PATIENT,0


Recorded 10 patient FK checks.


All clinical tables have zero orphan patient references. Every patient FKin the data resolves to a record in patients.Id.

### 7. Duplicate payer investigation
The PK check flagged 10 duplicate payer IDs across the union. Let's look at them. If the same payer ID is appearing once per state file, that's a Synthea generation pattern, not a data issue.

In [0]:
payers_df = spark.table(f"{bronze_prefix}.payers")

duplicate_payer_ids_df = (
    payers_df
    .groupBy("Id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

display(duplicate_payer_ids_df)

duplicate_payer_id_count = (
    spark.table(f"{bronze_prefix}.payers")
    .groupBy("Id")
    .count()
    .filter(col("count") > 1)
    .count()
)

run.add_result(
    check_id="bronze.payers.cross_state_duplicate_id",
    check_name="Cross-state duplicate payer IDs",
    check_category="pk_uniqueness",
    target_table="payers",
    target_column="Id",
    severity="warning",
    expected="0 (but acceptable >0 due to Synthea cross-state generation)",
    actual=str(duplicate_payer_id_count),
    status="warned" if duplicate_payer_id_count > 0 else "passed",
    message=f"{duplicate_payer_id_count} payer IDs appear in multiple states — same entity, expected pattern"
)

print(f"Recorded payer cross-state duplicate check: {duplicate_payer_id_count} duplicate IDs.")

Id,count
a735bf55-83e9-331a-899d-a82a60b9f60c,5
d18ef2e6-ef40-324c-be54-34a5ee865625,5
df166300-5a78-3502-a46a-832842197811,5
b046940f-1664-3047-bca7-dfa76be352a4,5
734afbd6-4794-363b-9bc0-6a3981533ed5,5
e03e23c9-4df1-3eb6-a62d-f70f02301496,5
26aab0cd-6aba-3e1b-ac5b-05c8867e762c,5
8fa6c185-e44e-3e34-8bd8-39be8694f4ce,5
0133f751-9229-3cfd-815f-b6d4979bdd6a,5
d31fccc3-1767-390d-966a-22a5156f4219,5


Recorded payer cross-state duplicate check: 10 duplicate IDs.


In [0]:
# Pull the actual duplicate records so we can confirm they're the same entity
# across states (not different payers that collided on Id)
duplicate_payer_records_df = (
    payers_df
    .join(
        duplicate_payer_ids_df.select("Id"),
        on="Id",
        how="inner"
    )
    .orderBy("Id")
)

display(duplicate_payer_records_df)

Id,NAME,OWNERSHIP,ADDRESS,CITY,STATE_HEADQUARTERED,ZIP,PHONE,AMOUNT_COVERED,AMOUNT_UNCOVERED,REVENUE,COVERED_ENCOUNTERS,UNCOVERED_ENCOUNTERS,COVERED_MEDICATIONS,UNCOVERED_MEDICATIONS,COVERED_PROCEDURES,UNCOVERED_PROCEDURES,COVERED_IMMUNIZATIONS,UNCOVERED_IMMUNIZATIONS,UNIQUE_CUSTOMERS,QOLS_AVG,MEMBER_MONTHS,source_state
0133f751-9229-3cfd-815f-b6d4979bdd6a,Aetna,PRIVATE,null,null,null,null,null,103242032.11,56824152.09,17379959.79,37856,0,24105,0,86430,0,12185,0,1296,0.9021323446057921,125016,California
0133f751-9229-3cfd-815f-b6d4979bdd6a,Aetna,PRIVATE,null,null,null,null,null,210622630.10,138521988.52,19872059.81,81784,0,44597,0,189354,0,26445,0,2474,0.8514309031317929,302292,Florida
0133f751-9229-3cfd-815f-b6d4979bdd6a,Aetna,PRIVATE,null,null,null,null,null,123116087.63,85029229.83,15311574.59,55945,0,27914,0,127799,0,19936,0,1780,0.8678394697588391,209184,Texas
0133f751-9229-3cfd-815f-b6d4979bdd6a,Aetna,PRIVATE,null,null,null,null,null,60440706.24,40424914.11,11290987.05,30519,0,15559,0,68032,0,10601,0,882,0.9249345033462824,105048,Pennsylvania
0133f751-9229-3cfd-815f-b6d4979bdd6a,Aetna,PRIVATE,null,null,null,null,null,30874829.45,16976051.61,6392999.17,10456,0,4290,0,23740,0,4579,0,455,0.9353162255954506,38796,New York
26aab0cd-6aba-3e1b-ac5b-05c8867e762c,Humana,PRIVATE,null,null,null,null,null,174297392.47,95127463.24,81484236.00,68752,0,28731,0,157869,0,26441,0,689,0.954363755444624,240420,Texas
26aab0cd-6aba-3e1b-ac5b-05c8867e762c,Humana,PRIVATE,null,null,null,null,null,153065323.19,79252825.27,62356860.00,52018,0,23537,0,119051,0,17608,0,439,0.9580158737308753,183612,New York
26aab0cd-6aba-3e1b-ac5b-05c8867e762c,Humana,PRIVATE,null,null,null,null,null,416387713.12,220271323.62,176767146.00,139597,0,57130,0,322898,0,56390,0,1401,0.9583651137001662,521436,California
26aab0cd-6aba-3e1b-ac5b-05c8867e762c,Humana,PRIVATE,null,null,null,null,null,285360460.78,141586177.39,108632880.00,88351,0,36712,0,203136,0,32430,0,842,0.9557322690888611,320172,Florida
26aab0cd-6aba-3e1b-ac5b-05c8867e762c,Humana,PRIVATE,null,null,null,null,null,126911279.36,73952426.50,66700260.00,56621,0,26732,0,130266,0,20854,0,554,0.9517277985708934,196776,Pennsylvania


### 8. Duplicate organization investigation
Same pattern check for organizations. The PK check flagged 13 duplicates.

In [0]:
organizations_df = spark.table(f"{bronze_prefix}.organizations")

duplicate_organization_ids_df = (
    organizations_df
    .groupBy("Id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

display(duplicate_organization_ids_df)

duplicate_org_id_count = (
    spark.table(f"{bronze_prefix}.organizations")
    .groupBy("Id")
    .count()
    .filter(col("count") > 1)
    .count()
)

run.add_result(
    check_id="bronze.organizations.cross_state_duplicate_id",
    check_name="Cross-state duplicate organization IDs",
    check_category="pk_uniqueness",
    target_table="organizations",
    target_column="Id",
    severity="warning",
    expected="0 (but acceptable >0 due to Synthea cross-state generation)",
    actual=str(duplicate_org_id_count),
    status="warned" if duplicate_org_id_count > 0 else "passed",
    message=f"{duplicate_org_id_count} organization IDs appear in multiple states — Synthea artifact, expected pattern"
)

print(f"Recorded organization cross-state duplicate check: {duplicate_org_id_count} duplicate IDs.")

Id,count
6234ba2c-6495-383e-b182-239bc16f2a99,4
18b1bb7e-2f93-3253-a7ae-d19c13ffeaa1,4
d2e92a91-1e9e-3b9b-92b1-d3fb3b3d57ed,3
213242fe-37b2-35bf-9370-0b5dda73832d,3
777bc54d-aa8c-377a-90af-04a6bd8fb7a9,2
23b6e874-653e-3558-b825-b07a725e5f14,2
f4c3453d-0dd5-320a-a319-33af27c4c9d0,2
a4fbc78f-6b9c-306c-9a50-01feadeceab9,2
886f436a-3c55-3c1d-b1d5-88e72a46cfbc,2
20a60f91-b2f2-3bf1-88f8-9912ac670f01,2


Recorded organization cross-state duplicate check: 13 duplicate IDs.


In [0]:
duplicate_organization_records_df = (
    organizations_df
    .join(
        duplicate_organization_ids_df.select("Id"),
        on="Id",
        how="inner"
    )
    .orderBy("Id")
)

display(duplicate_organization_records_df)

Id,NAME,ADDRESS,CITY,STATE,ZIP,LAT,LON,PHONE,REVENUE,UTILIZATION,source_state
18b1bb7e-2f93-3253-a7ae-d19c13ffeaa1,GARDEN STATE HEALTHCARE ASSOCIATES LLC,29 E 29TH ST,SANTA MONICA,CA,904022515,34.0220868,-118.4571734,2018587394,0.0,81,California
18b1bb7e-2f93-3253-a7ae-d19c13ffeaa1,GARDEN STATE HEALTHCARE ASSOCIATES LLC,29 E 29TH ST,GRAND PRAIRIE,TX,750545510,32.7459645,-96.9977846,2018587394,0.0,73,Texas
18b1bb7e-2f93-3253-a7ae-d19c13ffeaa1,GARDEN STATE HEALTHCARE ASSOCIATES LLC,29 E 29TH ST,FORT LAUDERDALE,FL,333061612,26.1223084,-80.1433786,2018587394,0.0,64,Florida
18b1bb7e-2f93-3253-a7ae-d19c13ffeaa1,GARDEN STATE HEALTHCARE ASSOCIATES LLC,29 E 29TH ST,SAINT JAMES,NY,117801655,40.8789871,-73.1567778,2018587394,0.0,94,New York
20a60f91-b2f2-3bf1-88f8-9912ac670f01,APPALACHIAN REGIONAL HEALTHCARE INC.,200 MEDICAL CENTER DR,GAINESVILLE,FL,326084146,29.6519684,-82.3249846,6064396600,0.0,15,Florida
20a60f91-b2f2-3bf1-88f8-9912ac670f01,APPALACHIAN REGIONAL HEALTHCARE INC.,200 MEDICAL CENTER DR,LATROBE,PA,156501609,40.317287,-79.3840301,6064396600,0.0,275,Pennsylvania
213242fe-37b2-35bf-9370-0b5dda73832d,ENLOE MEDICAL CENTER,1600 ESPLANADE,NEW YORK,NY,100165496,40.855734,-73.8587136,5303327300,0.0,215,New York
213242fe-37b2-35bf-9370-0b5dda73832d,ENLOE MEDICAL CENTER,1600 ESPLANADE,SAN DIEGO,CA,921271034,32.9896231,-117.1669814,5303327300,0.0,1577,California
213242fe-37b2-35bf-9370-0b5dda73832d,ENLOE MEDICAL CENTER,1600 ESPLANADE,VENETIA,PA,153671342,40.2467368,-80.042275,5303327300,0.0,50,Pennsylvania
23b6e874-653e-3558-b825-b07a725e5f14,PROVIDENCE HEALTH & SERVICES WASHINGTON,3200 PROVIDENCE DR,GREAT NECK,NY,110232449,40.8006567,-73.7284647,9072126002,0.0,20,New York


Same story as payers: each duplicate ID appears once per source state. Syntheagenerates the same organization entity in multiple state files. Bronze keepsthe rows; silver will dedupe using `(source_state, Id)` as the composite key.

### Bronze validation summary
Overall result:
- bronze passed.
- Warnings (all documented above, all handled in silver):
1. `observations.ENCOUNTER` has 316,980 nulls (QALY/DALY/QOLS lifetime metrics)
2. `payer_transitions.MEMBERID` has 228,151 nulls (self-pay enrollments)
3. `organizations.Id` has 13 cross-state duplicates (same entity in multiple files)
4. `payers.Id` has 10 cross-state duplicates (same)

Next: silver layer.

### Persist results and assert
Write everything to `_metadata.validation_results` + `_metadata.validation_runs`.The final `assert_no_failures()` raises if any error-level check failed. That's what stops the pipeline from continuing on bad data.

In [0]:
summary = run.commit()

print(f"Run ID:         {summary['run_id']}")
print(f"Status:         {summary['status']}")
print(f"Total checks:   {summary['total']}")
print(f"  Passed:       {summary['passed']}")
print(f"  Warned:       {summary['warned']}")
print(f"  Failed:       {summary['failed']}")
print(f"Duration:       {summary['duration_seconds']:.2f}s")

Run ID:         df777604-3d36-4aa5-b7cf-34e71e37c292
Status:         passed_with_warnings
Total checks:   67
  Passed:       61
  Warned:       6
  Failed:       0
Duration:       51.44s


In [0]:
run.assert_no_failures()

In [0]:
display(spark.table("healthcare_demo._metadata.validation_results").limit(20))

run_id,check_id,check_name,check_category,target_table,target_column,severity,expected,actual,status,message,check_timestamp
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.dim_patient.non_empty,Gold table dim_patient must be non-empty,row_count,dim_patient,null,error,>0,22817,passed,,2026-05-13T22:44:08.235Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.dim_provider.non_empty,Gold table dim_provider must be non-empty,row_count,dim_provider,null,error,>0,15449,passed,,2026-05-13T22:44:08.235Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.dim_organization.non_empty,Gold table dim_organization must be non-empty,row_count,dim_organization,null,error,>0,15430,passed,,2026-05-13T22:44:08.235Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.dim_payer.non_empty,Gold table dim_payer must be non-empty,row_count,dim_payer,null,error,>0,10,passed,,2026-05-13T22:44:08.236Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.dim_state.non_empty,Gold table dim_state must be non-empty,row_count,dim_state,null,error,>0,5,passed,,2026-05-13T22:44:08.236Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.dim_date.non_empty,Gold table dim_date must be non-empty,row_count,dim_date,null,error,>0,1826,passed,,2026-05-13T22:44:08.236Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.fact_encounters.non_empty,Gold table fact_encounters must be non-empty,row_count,fact_encounters,null,error,>0,459785,passed,,2026-05-13T22:44:08.236Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.fact_claims.non_empty,Gold table fact_claims must be non-empty,row_count,fact_claims,null,error,>0,850177,passed,,2026-05-13T22:44:08.236Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.bridge_claim_diagnosis.non_empty,Gold table bridge_claim_diagnosis must be non-empty,row_count,bridge_claim_diagnosis,null,error,>0,1240662,passed,,2026-05-13T22:44:08.236Z
ed780a13-3cf2-4873-8e65-dc36aa244354,gold.fact_patient_utilization.non_empty,Gold table fact_patient_utilization must be non-empty,row_count,fact_patient_utilization,null,error,>0,22817,passed,,2026-05-13T22:44:08.236Z
